In [5]:
import pandas as pd
import numpy as np
import altair as alt
from ecostyles import EcoStyles

styles = EcoStyles(); styles.register_and_enable_theme()

# history (paste full series) + projection, as ONE series so the axis rescales together
rows = """1900,37.8
1914,42
1916,103.1
1919,139.2
1921,174.5
1930,178.3
1939,136.6
1944,233.3
1946,251.7
1950,185.6
1960,102.4
1970,54.7
1980,40.3
1990,21.6
2000,28.2
2008,50.2
2010,70.6
2020,95.4
2025,93.8"""
hist = pd.DataFrame([r.split(",") for r in rows.strip().split("\n")], columns=["year","value"])
hist["year"]=hist["year"].astype(int); hist["value"]=hist["value"].astype(float)
hist["seg"]="History"

# OBR worst-case projection (use your exact FRS 5.3 annual values)
yrs=list(range(2026,2076))
proj_v=np.interp(yrs,[2026,2035,2045,2055,2065,2075],[95,131,207,334,563,1019])
proj=pd.DataFrame({"year":yrs,"value":proj_v,"seg":"Projection"})

df=pd.concat([hist,proj],ignore_index=True)
df["date"]=pd.to_datetime(df["year"],format="%Y")

# slider — drives BOTH the filter and (via auto-scale) the axis
slider=alt.binding_range(min=2025,max=2075,step=1,name="Show to year: ")
endYear=alt.param(name="endYear",value=2025,bind=slider)

base=alt.Chart(df).transform_filter(alt.datum.year<=endYear)

# history line (ink) + projection line (red dashed), NO fixed y-domain -> auto-rescales
hist_line=base.transform_filter(alt.datum.seg=="History").mark_line(
    strokeWidth=1.6,color="#122b39").encode(
    x=alt.X("date:T",axis=alt.Axis(format="%Y",tickCount=8),title=None),
    y=alt.Y("value:Q",scale=alt.Scale(zero=True),title="Net debt (% of GDP)",
            axis=alt.Axis(labelExpr="datum.label + '%'")))
proj_line=base.transform_filter(alt.datum.seg=="Projection").mark_line(
    strokeWidth=1.8,strokeDash=[4,3],color="#E6224B").encode(
    x="date:T",y="value:Q")

caption=alt.Title(
    text="Source: ONS / OBR (July 2026 Fiscal Risks and Sustainability report)",
    subtitle=["UK net debt, % of GDP. Drag to reveal the OBR projection — watch the axis rescale.",
              "As the projection climbs, the world wars shrink to blips."],
    orient="bottom",anchor="start",fontSize=11,subtitleFontSize=10,
    color="#676A86",subtitleColor="#676A86",dy=12)

chart=((hist_line+proj_line).add_params(endYear)
    .properties(width=700,height=360,title=caption)
    .configure(background="white",font="Circular Std")
    .configure_view(fill="transparent",stroke="transparent"))

chart   # display live; screen-record the drag for your video

alt.LayerChart(...)